# 05 Gradio 接口规范

## 1. 可视化系统概述

### 1.1 系统目标

根据 PDF 第二阶段"Python 综合系统设计"要求，构建一个交互式可视化系统，允许用户：
- 选择和组合数据处理方式
- 选择不同的模型（传统/CNN/无监督）
- 配置损失函数和超参数
- 选择验证方法
- 查看最终分类结果和对比

系统以 Gradio 为前端框架，采用 **5-Tab 布局**。本 Notebook 定义所有回调函数的接口规范，为后续可视化系统开发提供统一的对接标准。

### 1.2 5-Tab 布局设计

```
┌─────────────────────────────────────────────────┐
│  Tab 1: 数据处理  │ Tab 2: 模型选择 │ Tab 3: 损失函数 │
│  Tab 4: 参数优化  │ Tab 5: 模型验证               │
└─────────────────────────────────────────────────┘
```

| Tab | 名称 | 对应回调函数 | 功能 |
|-----|------|-------------|------|
| 1 | 数据处理 | `data_processing_handler()` | 选择划分策略、预处理方法、特征类型 |
| 2 | 模型选择 | `traditional_model_handler()` / `cnn_handler()` / `unsupervised_handler()` | 选择模型类型并配置 |
| 3 | 损失函数 | `cnn_handler(loss_fn=...)` | 选择损失函数（仅 CNN） |
| 4 | 参数优化 | 各 handler 的 `params` 参数 | 配置超参数 |
| 5 | 模型验证 | `compare_all_models_handler()` | 综合对比所有结果 |

## 2. 接口规范总览

| 接口函数 | 所属 Notebook | 输入参数 | 输出类型 |
|----------|:--:|------|------|
| `data_processing_handler` | 01 | split_method, split_ratio, k_folds, use_stratify, preprocessing, features | dict (comparison_table, best_pipeline, plots) |
| `traditional_model_handler` | 02 | model_type, preprocessing, features, model_params, cv_folds | dict (metrics, confusion_matrix, roc_curve, cv_results) |
| `cnn_handler` | 03 | preprocessing, loss_fn, focal_alpha/gamma, optimizer, lr, dropout, batch_size, epochs | dict (metrics, train_history, confusion_matrix, roc_curve, best_hyperparams) |
| `unsupervised_handler` | 04 | method, n_clusters, feature_set, method_params | dict (labels, internal_metrics, external_metrics, cluster_plot) |
| `compare_all_models_handler` | 05 | model_configs, validation_method | dict (comparison_table, roc_curves, summary) |

## 3. 数据处理接口规范

### `data_processing_handler`

**来源**：`01_数据处理与特征工程.ipynb` 第 10 节

```
def data_processing_handler(
    split_method: str = "holdout",   # "holdout" | "kfold"
    split_ratio: float = 0.7,        # 留出法训练集比例
    k_folds: int = 5,                # K折数
    use_stratify: bool = True,       # 是否分层抽样
    preprocessing: list = None,      # ["none","clahe","gaussian","median","clahe+gaussian","clahe+median"]
    features: list = None,           # ["hog","lbp","glcm","edge_density"]
    max_samples: int = 2000,         # 样本上限
    random_seed: int = 42,           # 随机种子
) -> dict                            # {comparison_table, best_pipeline, best_features, ...}
```

## 4. 传统模型接口规范

### `traditional_model_handler`

**来源**：`02_传统监督学习对比.ipynb` 第 14 节

```
def traditional_model_handler(
    model_type: str = "random_forest",  # "decision_tree"|"svm"|"naive_bayes"|"random_forest"|"logistic_regression"|"xgboost"|"lightgbm"
    preprocessing: list = None,         # 预处理方法
    features: list = None,              # 特征类型
    model_params: dict = None,          # {"n_estimators":100,"max_depth":20,...}
    cv_folds: int = 5,                  # 交叉验证折数
    max_samples: int = 2000,            # 样本上限
    random_seed: int = 42,              # 随机种子
) -> dict                               # {model_name, metrics, confusion_matrix, roc_curve, cv_results, feature_importance, training_time}
```

### 各模型支持的参数

| 模型 | 关键参数 |
|------|------|
| 决策树 | max_depth, criterion, min_samples_split |
| SVM | kernel, C, gamma |
| 朴素贝叶斯 | var_smoothing |
| 随机森林 | n_estimators, max_depth, min_samples_split |
| 逻辑回归 | C, penalty, solver |
| XGBoost | n_estimators, max_depth, learning_rate, subsample |
| LightGBM | n_estimators, max_depth, learning_rate, num_leaves |

## 5. CNN 接口规范

### `cnn_handler`

**来源**：`03_深度学习对比.ipynb` 第 11 节

这是可视化系统中参数最多的核心接口，覆盖模型选择、损失函数、参数优化三个 Tab。

```
def cnn_handler(
    preprocessing: list = None,              # 预处理管线
    loss_fn: str = "cross_entropy",          # "cross_entropy"|"focal"|"label_smoothing"|"dice"
    focal_alpha: float = 0.25,               # Focal Loss alpha (0~1)
    focal_gamma: float = 2.0,                # Focal Loss gamma (0~5)
    label_smoothing_epsilon: float = 0.1,    # Label Smoothing epsilon
    optimizer_name: str = "adam",            # "adam"|"sgd"
    learning_rate: float = 1e-3,             # 学习率
    dropout_rate: float = 0.5,               # Dropout 比例
    batch_size: int = 64,                    # 批量大小
    epochs: int = 30,                        # 最大训练轮数
    early_stopping_patience: int = 10,       # 早停耐心值
    max_samples: int = 4000,                 # 样本上限
    random_seed: int = 42,                   # 随机种子
) -> dict                                    # {metrics, train_history, confusion_matrix, roc_curve, best_hyperparams, model_weights_path}
```

### 返回值详情

| 字段 | 类型 | 说明 |
|------|------|------|
| metrics | dict | 测试集评估指标 (accuracy, precision, recall, f1, roc_auc) |
| train_history | dict | 训练曲线数据 {train_loss, val_loss, train_acc, val_acc} |
| confusion_matrix | plt.Figure | 混淆矩阵图 |
| roc_curve | plt.Figure | ROC 曲线图 |
| best_hyperparams | dict | 实际使用的超参数组合 |
| model_weights_path | str | 模型权重保存路径 |

## 6. 无监督接口规范

### `unsupervised_handler`

**来源**：`04_无监督学习对比.ipynb` 第 5 节

```
def unsupervised_handler(
    method: str = "kmeans",            # "kmeans"|"gmm"|"dbscan"|"agglomerative"|"spectral"
    n_clusters: int = 2,               # 聚类数量
    feature_set: list = None,          # 特征子集
    method_params: dict = None,        # 方法特定参数
    max_samples: int = 2000,           # 样本上限
    random_seed: int = 42,             # 随机种子
) -> dict                              # {labels, internal_metrics, external_metrics, cluster_plot, method_summary}
```

### 各方法默认参数

| 方法 | 默认参数 | 特殊说明 |
|------|------|------|
| K-Means | n_clusters=2, random_state=42, n_init='auto' | — |
| GMM | n_components=2, random_state=42 | 支持概率输出 |
| DBSCAN | eps=0.5, min_samples=5 | 噪声点标签=-1 |
| 层次聚类 | n_clusters=2, linkage='ward' | — |
| 谱聚类 | n_clusters=2, affinity='rbf', random_state=42 | 对非凸形状数据有效 |

## 7. 接口组合示例

### 端到端对比实验流程

```python
# 示例 1：对比不同预处理对随机森林的影响
data_result = data_processing_handler(
    split_method="holdout", split_ratio=0.7,
    preprocessing=["clahe", "median"],
    features=["hog", "lbp", "glcm", "edge_density"]
)
model_result = traditional_model_handler(
    model_type="random_forest",
    model_params={"n_estimators": 100, "max_depth": 20}
)

# 示例 2：CNN 完整训练并对比损失函数
cnn_result_ce = cnn_handler(
    preprocessing=["clahe", "median"],
    loss_fn="cross_entropy",
    learning_rate=1e-3, epochs=30
)
cnn_result_focal = cnn_handler(
    preprocessing=["clahe", "median"],
    loss_fn="focal", focal_alpha=0.25, focal_gamma=2.0,
    learning_rate=1e-3, epochs=30
)

# 示例 3：无监督聚类探索
unsup_result = unsupervised_handler(
    method="kmeans", n_clusters=2,
    feature_set=["hog", "lbp", "glcm", "edge_density"]
)
```

## 8. 跨模型对比接口（完整实现）

### `compare_all_models_handler`

统一的跨模型对比入口，用于 Tab 5 "模型验证"。支持比较传统模型、CNN 和无监督方法。

```python
def compare_all_models_handler(
    model_configs: list = None,
    data_config: dict = None,
    validation_method: str = "holdout",
    max_samples: int = 1000,
    random_seed: int = 42,
) -> dict:
    """统一跨模型对比接口。

    Parameters
    ----------
    model_configs : list
        模型配置列表，每个元素为 {"type": str, "params": dict}。
        type 可选: "decision_tree","svm","naive_bayes","random_forest",
        "logistic_regression","xgboost","lightgbm","cnn",
        "kmeans","gmm","dbscan","agglomerative","spectral"。
    data_config : dict or None
        数据处理配置。None 则使用默认管线。
    validation_method : str
        "holdout"|"kfold"。
    max_samples : int
        每类样本数上限。
    random_seed : int
        随机种子。

    Returns
    -------
    dict: {comparison_table, roc_curves, best_model, best_metrics, summary_text}
    """
    # Actual implementation below
```


In [ ]:
# ========================================
# 7.5 统一数据管线：prepare_data()
# ========================================
# 供 compare_all_models_handler 等函数使用，
# 确保所有模型在完全相同的数据上对比。

def prepare_data(
    max_samples: int = 1000,
    random_seed: int = 42,
    split_method: str = "holdout",
    split_ratio: float = 0.7,
    preprocessing: list = None,
    features: list = None,
    **kwargs,
) -> dict:
    """统一数据管线：加载→预处理→提特征→划分，返回训练/测试集。

    Parameters
    ----------
    max_samples : int  每类样本数上限（总数≈max_samples×2）。
    random_seed : int  随机种子。
    split_method : str  "holdout"|"kfold"。
    split_ratio : float 留出法训练集比例。
    preprocessing : list  预处理方法列表，如 ["clahe","median"]。
    features : list  特征类型列表，如 ["hog","lbp","glcm","edge_density"]。
    kwargs : dict  传给 data_processing_handler 的额外参数。

    Returns
    -------
    dict: {
        "X_train": np.ndarray, "X_test": np.ndarray,
        "y_train": np.ndarray, "y_test": np.ndarray,
        "raw_images": np.ndarray, "raw_labels": np.ndarray,
        "config": dict,  # 实际使用的管线参数
    }
    """
    import numpy as np
    from sklearn.model_selection import train_test_split

    if preprocessing is None:
        preprocessing = ["clahe", "median"]
    if features is None:
        features = ["hog", "lbp", "glcm", "edge_density"]

    # 加载原始图像
    n_per_class = max_samples // 2
    images, labels = load_dataset(DATA_ROOT, max_samples=n_per_class)

    # 预处理管线组合
    pipeline_map = {
        "none": lambda img: img,
        "clahe": lambda img: apply_clahe(img),
        "gaussian": lambda img: apply_gaussian_filter(img),
        "median": lambda img: apply_median_filter(img),
        "clahe+gaussian": lambda img: apply_gaussian_filter(apply_clahe(img)),
        "clahe+median": lambda img: apply_median_filter(apply_clahe(img)),
    }

    def compose_preprocess(img):
        for p in preprocessing:
            if p in pipeline_map and p != "none":
                img = pipeline_map[p](img)
        return img

    # 特征提取映射
    feat_map = {
        "hog": extract_hog_features,
        "lbp": extract_lbp_features,
        "glcm": extract_glcm_features,
        "edge_density": lambda img: np.array([extract_edge_density(img)]),
    }

    def extract_selected(img):
        parts = []
        for f in features:
            if f in feat_map:
                parts.append(feat_map[f](img))
        return np.concatenate(parts)

    # 预处理 + 特征提取
    processed = np.array([compose_preprocess(img) for img in images])
    X_all = np.stack([extract_selected(img) for img in processed])
    y_all = labels

    # 划分
    test_size = round(1.0 - split_ratio, 4)
    X_train, X_test, y_train, y_test = train_test_split(
        X_all, y_all, test_size=test_size,
        random_state=random_seed, stratify=y_all,
    )

    print(f"prepare_data: {len(y_train)} train / {len(y_test)} test, "
          f"dim={X_train.shape[1]}, preproc={preprocessing}, feats={features}")

    return {
        "X_train": X_train, "X_test": X_test,
        "y_train": y_train, "y_test": y_test,
        "raw_images": images, "raw_labels": labels,
        "config": {
            "preprocessing": preprocessing,
            "features": features,
            "split_method": split_method,
            "split_ratio": split_ratio,
            "n_samples": len(labels),
            "feature_dim": X_train.shape[1],
        },
    }


print("prepare_data() — 统一数据管线已定义。")
print("用法: data = prepare_data(max_samples=1000, preprocessing=['clahe','median'], features=['hog','lbp','glcm','edge_density'])")

In [ ]:
def compare_all_models_handler(
    model_configs: list = None,
    data_config: dict = None,
    validation_method: str = "holdout",
    max_samples: int = 1000,
    random_seed: int = 42,
) -> dict:
    """统一跨模型对比接口 — Tab 5 "模型验证"。

    使用 prepare_data() 统一数据管线，确保所有模型在完全相同的数据上对比。
    支持传统模型（加载预训练）、CNN（加载预训练）、无监督方法。

    Parameters
    ----------
    model_configs : list
        [{"type":"random_forest"}, {"type":"cnn","params":{...}}, ...]
    data_config : dict or None
        传递给 prepare_data() 的参数。None 使用默认。
    validation_method : str
        "holdout" | "kfold"
    max_samples : int
        样本上限。
    random_seed : int
        随机种子。

    Returns
    -------
    dict: {comparison_table, roc_curves, pr_curves, radar_chart, best_model, best_metrics, summary_text}
    """
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt

    if model_configs is None:
        model_configs = [
            {"type": "random_forest"},
            {"type": "xgboost"},
            {"type": "lightgbm"},
            {"type": "svm"},
        ]

    # 构建数据管线参数
    prep_kwargs = {"max_samples": max_samples, "random_seed": random_seed}
    if data_config is not None:
        prep_kwargs.update(data_config)
    prep_kwargs.setdefault("preprocessing", ["clahe", "median"])
    prep_kwargs.setdefault("features", ["hog", "lbp", "glcm", "edge_density"])
    prep_kwargs.setdefault("split_method", validation_method)

    # ===== 统一数据管线：只执行一次 =====
    print("准备共享数据管线...")
    data = prepare_data(**prep_kwargs)
    X_tr = data["X_train"]; X_te = data["X_test"]
    y_tr = data["y_train"]; y_te = data["y_test"]

    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

    all_metrics = []
    roc_data = []

    # ===== 逐一评估每个模型（共享同一份数据）=====
    for cfg in model_configs:
        model_type = cfg.get("type", "random_forest")
        params = cfg.get("params", {})
        print(f"  对比评估: {model_type}...")

        try:
            if model_type in ("decision_tree", "svm", "naive_bayes", "random_forest",
                              "logistic_regression", "xgboost", "lightgbm"):
                # 调用 traditional_model_handler（传入共享数据）
                result = traditional_model_handler(
                    model_type=model_type,
                    model_params=params.get("model_params"),
                    cv_folds=3,
                    max_samples=max_samples,
                    random_seed=random_seed,
                    prepared_data=data,  # ← 关键：共享数据
                )
                m = result["metrics"]
                all_metrics.append({
                    "模型": model_type,
                    "准确率": m["accuracy"], "F1分数": m["f1"],
                    "ROC-AUC": m["roc_auc"],
                })
                # 获取概率用于 ROC
                if "roc_curve" in result:
                    from sklearn.metrics import roc_auc_score
                    # 用 pretrained model 重新获取 proba
                    try:
                        import joblib
                        trad_dir = Path.cwd().parent / "outputs" / "models" / "traditional"
                        mp = trad_dir / f"{model_type}_best.joblib"
                        if mp.exists():
                            model = joblib.load(mp)
                            model.fit(X_tr, y_tr)
                            try:
                                y_prob = model.predict_proba(X_te)[:, 1]
                                roc_data.append((model_type, y_prob))
                            except Exception:
                                pass
                    except Exception:
                        pass

            elif model_type == "cnn":
                # 调用 cnn_handler（传入共享数据）
                cnn_result = cnn_handler(
                    preprocessing=data["config"]["preprocessing"],
                    loss_fn=params.get("loss_fn", "cross_entropy"),
                    learning_rate=params.get("learning_rate", 1e-3),
                    epochs=params.get("epochs", 10),
                    max_samples=max_samples,
                    random_seed=random_seed,
                    prepared_data=data,  # ← 共享数据
                )
                all_metrics.append({
                    "模型": f"CNN ({params.get('loss_fn', 'CE')})",
                    "准确率": cnn_result["metrics"]["accuracy"],
                    "F1分数": cnn_result["metrics"]["f1"],
                    "ROC-AUC": cnn_result["metrics"]["roc_auc"],
                })

            elif model_type in ("kmeans", "gmm", "dbscan", "agglomerative", "spectral"):
                unsup_result = unsupervised_handler(
                    method=model_type,
                    n_clusters=params.get("n_clusters", 2),
                    max_samples=max_samples,
                    random_seed=random_seed,
                    prepared_data=data,  # ← 共享数据
                )
                ext = unsup_result.get("external_metrics", {})
                all_metrics.append({
                    "模型": model_type,
                    "ARI": ext.get("ari"),
                    "F1分数": None,
                    "ROC-AUC": None,
                })

        except Exception as e:
            print(f"    {model_type} 评估失败: {e}")
            all_metrics.append({"模型": model_type, "错误": str(e)})

    # ===== 生成对比结果 =====
    import pandas as pd
    comparison_table = pd.DataFrame(all_metrics)

    # ROC 叠加图
    fig_roc = None
    if roc_data:
        from sklearn.metrics import roc_curve, roc_auc_score
        fig_roc, ax_roc = plt.subplots(figsize=(8, 6))
        for name, y_prob in roc_data:
            fpr, tpr, _ = roc_curve(y_te, y_prob)
            ax_roc.plot(fpr, tpr, linewidth=2, label=f"{name} (AUC={roc_auc_score(y_te, y_prob):.4f})")
        ax_roc.plot([0,1], [0,1], 'gray', linestyle='--')
        ax_roc.set_xlabel("假阳性率"); ax_roc.set_ylabel("真阳性率")
        ax_roc.set_title("多模型 ROC 曲线叠加（同一测试集）")
        ax_roc.legend(); ax_roc.grid(True, alpha=0.3)
        plt.tight_layout()

    # PR 叠加图
    fig_pr_all = None
    if roc_data:
        from sklearn.metrics import precision_recall_curve, average_precision_score
        fig_pr_all, ax_pr_all = plt.subplots(figsize=(8, 6))
        for name, y_prob in roc_data:
            precisions, recalls, _ = precision_recall_curve(y_te, y_prob)
            ap = average_precision_score(y_te, y_prob)
            ax_pr_all.plot(recalls, precisions, linewidth=2, label=f"{name} (AP={ap:.4f})")
        ax_pr_all.axhline(y=np.mean(y_te), color='gray', linestyle='--')
        ax_pr_all.set_xlabel("召回率"); ax_pr_all.set_ylabel("精确率")
        ax_pr_all.set_title("多模型 PR 曲线叠加（同一测试集）")
        ax_pr_all.legend(); ax_pr_all.grid(True, alpha=0.3)
        plt.tight_layout()

    # 雷达图
    fig_radar = None
    radar_cols = ["准确率", "F1分数"]
    valid_radar = comparison_table.dropna(subset=radar_cols)
    if len(valid_radar) >= 3:
        from math import pi
        N = len(radar_cols)
        angles = [n / float(N) * 2 * pi for n in range(N)]
        angles += angles[:1]
        fig_radar, ax_radar = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
        colors_r = plt.cm.tab10(np.linspace(0, 1, len(valid_radar)))
        for idx, (_, row) in enumerate(valid_radar.iterrows()):
            values = [row[c] for c in radar_cols] + [row[radar_cols[0]]]
            ax_radar.plot(angles, values, 'o-', linewidth=2, color=colors_r[idx], label=row["模型"])
            ax_radar.fill(angles, values, alpha=0.1, color=colors_r[idx])
        ax_radar.set_xticks(angles[:-1]); ax_radar.set_xticklabels(radar_cols)
        ax_radar.set_ylim(0.5, 1.0)
        ax_radar.set_title("多模型雷达图对比")
        ax_radar.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
        plt.tight_layout()

    # 最优模型
    if "F1分数" in comparison_table.columns:
        valid = comparison_table.dropna(subset=["F1分数"])
        if len(valid) > 0:
            best_row = valid.loc[valid["F1分数"].idxmax()]
            best_model_name = best_row["模型"]
            best_metrics = best_row.to_dict()
        else:
            best_model_name = "N/A"; best_metrics = {}
    else:
        best_model_name = "N/A"; best_metrics = {}

    pipeline_desc = f"预处理={data['config']['preprocessing']}, 特征={data['config']['features']}, 样本={data['config']['n_samples']}"
    summary = f"在统一数据管线（{pipeline_desc}）上对比 {len(model_configs)} 个模型。"
    if best_model_name != "N/A":
        summary += f" 最优模型: {best_model_name} (F1={best_metrics.get('F1分数', 'N/A')})。"

    return {
        "comparison_table": comparison_table,
        "roc_curves": fig_roc,
        "pr_curves": fig_pr_all,
        "radar_chart": fig_radar,
        "best_model": best_model_name,
        "best_metrics": best_metrics,
        "summary_text": summary,
    }


print("=" * 70)
print("compare_all_models_handler — 已重构")
print("=" * 70)
print("核心改进：使用 prepare_data() 统一数据管线")
print("所有模型在同一份 X_train/X_test 上评估，实现公平对比")
print("=" * 70)

## 9. 接口实现状态

| 接口函数 | 状态 | 实现位置 |
|----------|:--:|------|
| `data_processing_handler` | ✅ 已实现 | 01_数据处理与特征工程.ipynb §10 |
| `traditional_model_handler` | ✅ 已实现 | 02_传统监督学习对比.ipynb §14 |
| `cnn_handler` | ✅ 已实现 | 03_深度学习对比.ipynb §11 |
| `unsupervised_handler` | ✅ 已实现 | 04_无监督学习对比.ipynb §5 |
| `compare_all_models_handler` | ✅ 已实现 | 05_Gradio接口规范.ipynb §8 |

> ✅ = 已完整实现，可直接供可视化系统调用。

### 预训练模型清单

所有预训练模型已存储至 `outputs/models/` 目录：

| 类别 | 模型文件 | 数量 |
|------|----------|:--:|
| 传统监督 | `decision_tree`, `svm`, `naive_bayes`, `random_forest`, `logistic_regression`, `xgboost`, `lightgbm` | 7 |
| CNN | `cross_entropy`, `focal_default`, `focal_high_alpha`, `focal_high_gamma`, `label_smoothing`, `dice` | 6 |
| 无监督 | `kmeans`, `gmm`, `agglomerative`, `spectral` | 4 |
| **总计** | | **17 个模型** |

### 下一步

1. 创建 `src/gradio_app.py` 实现 Gradio Blocks 界面
2. 在各 Tab 中调用对应的 handler 函数
3. 测试界面交互体验
4. 录制操作视频